In [ ]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler


def add_pdays_flag(X):
    X = X.copy()
    X["pdays_never"] = (X["pdays"] == 999).astype(int)
    X["pdays"] = X["pdays"].replace(999, np.nan)   # ТАҢДА: NaN + median толтыру, не басқа жол?
    return X


def unknown_to_nan(X):
    X = X.copy()
    cat = X.select_dtypes(include="object").columns
    X[cat] = X[cat].replace("unknown", np.nan)
    return X


def build_preprocessor(unknown="category"):
    """unknown='category' -> 'unknown' бөлек санат; 'nan' -> NaN + most_frequent."""
    num = Pipeline([("imp", SimpleImputer(strategy="median")),
                    ("sc", StandardScaler())])
    if unknown == "category":
        cat = OneHotEncoder(handle_unknown="ignore")
        steps = [FunctionTransformer(add_pdays_flag)]
    else:
        cat = Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                        ("oh", OneHotEncoder(handle_unknown="ignore"))])
        steps = [FunctionTransformer(add_pdays_flag),
                 FunctionTransformer(unknown_to_nan)]
    ct = ColumnTransformer([
        ("num", num, make_column_selector(dtype_include=np.number)),
        ("cat", cat, make_column_selector(dtype_include=object)),
    ])
    return Pipeline(steps + [("ct", ct)])

# Task 3 · Midterm Project: Bank Marketing, term deposit
**Автор:** Baizhan ___ | **SEED:** 7301 

Notebook Restart & Run All арқылы жоғарыдан төмен орындалады.

In [ ]:
import sys, sklearn, numpy as np, pandas as pd, matplotlib.pyplot as plt, joblib
from sklearn.pipeline import Pipeline
from sklearn.model_selection import (train_test_split, StratifiedKFold,
                                     cross_validate, RandomizedSearchCV)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (roc_auc_score, average_precision_score, roc_curve,
                             precision_recall_curve, confusion_matrix)
from helpers import build_preprocessor

SEED = 0000   
print(sys.version); print("sklearn", sklearn.__version__,
      "numpy", np.__version__, "pandas", pd.__version__, "SEED", 7301)

## Stage 1 · Мәселені қою

**Бизнес сұрағы (өз сөзіммен).** Банк мерзімді депозит сату үшін қоңырау шалады. Бір толқында клиенттердің тек **10%-на** қоңырау шалуға болады. Модель клиенттерді "yes" ықтималдығы бойынша ___, сонда шақырылған 10% ішінде "yes" ең көп болуы керек.

**Метрика.** 10% шектеуі ___ болғандықтан, ранжирлеу сапасы маңызды: ROC-AUC, average precision, precision@10%, recall@10%, lift. Accuracy қолданылмайды, себебі ___.

**train мен test бағандарының айырмашылығы.** ___ . Себебі: ___ . Салдары: ___ бағаны фичаға кірмейді, өйткені ___.

In [ ]:
train = pd.read_csv("data/train.csv")
test = pd.read_csv("data/test.csv")
print(train.shape, test.shape)
print("тек train-де:", set(train.columns) - set(test.columns))
print("тек test-те:", set(test.columns) - set(train.columns))

## Stage 2 · EDA

**Баланс.** "yes" = ___ %. Әрқашан "no" дейтін модельдің accuracy-і = ___ → accuracy ___.

**"unknown" және сентинел (санымен).** Төмендегі кестеде: ___. `pdays=999` саны: ___ (___ %), мағынасы: бұрын байланыс болмаған.

**Қатар реті.** ___ (тексергенім: month ауысу саны ___). Қорытынды: ___.

**Графиктер.** 1) ___ бойынша "yes" үлесі: ___ санаты ___ %, ___ санаты ___ %. 2) ___: "yes" тобының медианасы ___, "no" тобыныкі ___.

In [ ]:
y = (train["y"] == "yes").astype(int)
print("yes үлесі:", y.mean(), "| majority accuracy:", 1 - y.mean())

obj = train.select_dtypes(exclude="number").columns.drop("y")
print((train[obj] == "unknown").sum().loc[lambda s: s > 0])
print("pdays==999:", (train["pdays"] == 999).sum(), (train["pdays"] == 999).mean())

print(train["month"].head(15).tolist())
print("month ауысу саны:", (train["month"] != train["month"].shift()).sum())

print(train.groupby("job")["y"].apply(lambda s: (s == "yes").mean()).sort_values())
print(train.groupby("y")["euribor3m"].median())      # ТАҢДА: өз фичаларыңды

train.groupby("poutcome")["y"].apply(lambda s: (s == "yes").mean()).plot.bar()
plt.ylabel("yes үлесі"); plt.title("poutcome бойынша yes үлесі"); plt.show()
train.boxplot(column="euribor3m", by="y"); plt.show()

## Stage 3 · Валидация

`train.csv` → **dev** (80%) және **hold-out** (20%, Stage 7-ге дейін қолданылмайды). Барлық салыстыру мен тюнинг тек dev-те `StratifiedKFold(n_splits=__, shuffle=__, random_state=SEED)` арқылы.

- *Неге stratified:* "yes" ~11.3% ғана, фолдтарда үлес ауытқымауы үшін.
- *Неге __ фолд:* ___.
- *Shuffle:* ___ (қатар ретін тексергенім: ___).
- *Std ережесі:* екі модель орташасының айырмасы фолд std-інен кіші болса, жеңімпаз жарияламаймын.

leak_cols = ["y"] + [c for c in train.columns if c not in test.columns and c != "y"]
X = train.drop(columns=leak_cols)
X_dev, X_hold, y_dev, y_hold = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED)
print(X_dev.shape, X_hold.shape, y_dev.mean(), y_hold.mean())

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)  # ТАҢДА

def cv_scores(model, scoring=("roc_auc", "average_precision")):
    r = cross_validate(model, X_dev, y_dev, cv=cv, scoring=list(scoring), n_jobs=-1)
    return {m: (round(r[f"test_{m}"].mean(), 4), round(r[f"test_{m}"].std(), 4))
            for m in scoring}

## Stage 4 · Pipeline ішіндегі preprocessing

| Түрі | Бағандар | Өңдеу | Себебі |
|---|---|---|---|
| Сан | ___ | median толтыру + StandardScaler | ___ |
| Категориялық | ___ | OneHotEncoder(handle_unknown="ignore") | ___ |
| "unknown" | ___ | ___ | ___ |
| pdays=999 | pdays | индикатор `pdays_never` + NaN→median | ___ |

**Leakage-тан қорғау.** Барлық fit қадамдары `Pipeline` ішінде, `cross_validate` оларды әр фолдтың train бөлігінде ғана fit етеді. Дәлел: төмендегі ұяшықта фолд ішіндегі scaler орташасы бүкіл dev орташасынан өзгеше.

**Салыстыру (A: "unknown" бөлек санат, B: NaN + толтыру):** A = ___ ± ___, B = ___ ± ___. Айырма ___, std ___ → ___.

In [ ]:
base = Pipeline([("prep", build_preprocessor()),
                 ("clf", LogisticRegression(max_iter=2000))])

# Leakage дәлелі
tr_idx, _ = next(cv.split(X_dev, y_dev))
base.fit(X_dev.iloc[tr_idx], y_dev.iloc[tr_idx])
ct = base.named_steps["prep"].named_steps["ct"]
num_cols = ct.transformers_[0][2]
sc = ct.named_transformers_["num"].named_steps["sc"]
cmp = pd.DataFrame({
    "фолд train орташасы (scaler)": sc.mean_,
    "бүкіл dev орташасы": base.named_steps["prep"].named_steps["flag"]
        .transform(X_dev)[num_cols].mean().values}, index=num_cols)
print(cmp.head(8))   # бағандар сәл өзгеше → scaler тек фолд train-інде fit болды

# unknown: A vs B
for mode in ["category", "nan"]:
    p = Pipeline([("prep", build_preprocessor(mode)),
                  ("clf", LogisticRegression(max_iter=2000))])
    print(mode, cv_scores(p))

# Raw test бағандарын қабылдайтынын тексеру
print(base.predict_proba(test.drop(columns=["id"])[X.columns].head())[:, 1])

## Stage 5 · Baselines (бірдей фолдтар)

| Модель | Accuracy | ROC-AUC | Average precision |
|---|---|---|---|
| Dummy (majority/prior) | ___ ± ___ | ___ ± ___ | ___ ± ___ |
| LogisticRegression | ___ ± ___ | ___ ± ___ | ___ ± ___ |

Majority accuracy ___ болса да AUC ___ → accuracy адастырады. Келесі модельдердің бәрі осы деңгейден (logreg AUC ___ ± ___) асуы керек.

In [ ]:
dummy = Pipeline([("prep", build_preprocessor()),
                  ("clf", DummyClassifier(strategy="prior"))])
three = ("accuracy", "roc_auc", "average_precision")
print("dummy ", cv_scores(dummy, three))
print("logreg", cv_scores(base, three))

## Stage 6 · Модельдерді салыстыру және тюнинг

Отбасылар: LogisticRegression, DecisionTree, RandomForest, LinearSVC. Бірдей pipeline және бірдей фолдтар.

**Іздеу кеңістігі:** logreg `C` ∈ {0.01, 0.1, 1, 10}; tree `max_depth` ∈ {3,5,8,12}, `min_samples_leaf` ∈ {10,50,200}; rf `max_depth` ∈ {6,10,None}, `min_samples_leaf` ∈ {5,20,50}; svm `C` ∈ {0.01, 0.1, 1}. `RandomizedSearchCV`, метрика ROC-AUC, тек dev-те.

**Нәтиже:** (кесте төменгі ұяшықтан) ___.

**class_weight:** бар/жоқ айырмасы ___, std ___ → ___.

**Қорытынды:** ___ мен ___ арасындағы айырма ___ std-тен ___ → ___. Финалға таңдағаным: ___, себебі ___.

In [ ]:
def mk(clf):
    return Pipeline([("prep", build_preprocessor()), ("clf", clf)])

spaces = {   # ТАҢДА: кеңістікті өзгерте аласың
    "logreg": (mk(LogisticRegression(max_iter=2000)), {"clf__C": [0.01, 0.1, 1, 10]}),
    "tree": (mk(DecisionTreeClassifier(random_state=SEED)),
             {"clf__max_depth": [3, 5, 8, 12], "clf__min_samples_leaf": [10, 50, 200]}),
    "rf": (mk(RandomForestClassifier(n_estimators=200, random_state=SEED, n_jobs=-1)),
           {"clf__max_depth": [6, 10, None], "clf__min_samples_leaf": [5, 20, 50]}),
    "svm": (mk(LinearSVC(random_state=SEED)), {"clf__C": [0.01, 0.1, 1]}),
}
best, rows = {}, []
for name, (model, grid) in spaces.items():
    n_comb = int(np.prod([len(v) for v in grid.values()]))
    s = RandomizedSearchCV(model, grid, n_iter=min(6, n_comb), scoring="roc_auc",
                           cv=cv, random_state=SEED, n_jobs=1)
    s.fit(X_dev, y_dev)
    best[name] = s.best_estimator_
    rows.append({"model": name, "params": s.best_params_,
                 "AUC mean": s.best_score_,
                 "AUC std": s.cv_results_["std_test_score"][s.best_index_]})
results = pd.DataFrame(rows).sort_values("AUC mean", ascending=False)
print(results.to_string())

# class_weight бар/жоқ (ТАҢДА: басқа модельдерге де жаса)
for cw in [None, "balanced"]:
    m = sklearn.base.clone(best["logreg"]).set_params(clf__class_weight=cw)
    print("logreg class_weight =", cw, cv_scores(m))

## Stage 7 · Финалдық бағалау (hold-out, бір рет)

Таңдалған модель: ___. Dev-те fit етілді, hold-out-та **бір рет** бағаланды.

ROC-AUC = ___, Average precision = ___.

**10% ереже.** Hold-out ___ қатар → ___ клиент шақырылады. Шек = ___.
Precision@10% = ___, Recall@10% = ___, базалық "yes" үлесі = ___, **Lift = ___**.

**Confusion matrix (менеджерге):** ___ клиентке қоңырау шалдық, ___-і келісті (TP); ___ келісетін клиентті өткізіп жібердік (FN); ___ қоңырау босқа кетті (FP).

**Кездейсоқ бөлу нәтижені қалай асырады:** нақты қолданыста өткен деректен оқып, болашаққа қоңырау шаламыз; ал бұл жерде train/hold-out кездейсоқ бөлінген, сондықтан ___ (уақыт дрейфі, макроэкономикалық жағдай екі жақта да араласқан). Нақты көрсеткіш ___ болуы мүмкін.

In [ ]:
final = best["rf"]          # ТАҢДА: Stage 6 нәтижесіне сүйеніп таңда
final.fit(X_dev, y_dev)
s = final.predict_proba(X_hold)[:, 1] if hasattr(final, "predict_proba") \
    else final.decision_function(X_hold)

auc, ap = roc_auc_score(y_hold, s), average_precision_score(y_hold, s)
print("AUC", auc, "AP", ap)

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
fpr, tpr, _ = roc_curve(y_hold, s)
ax[0].plot(fpr, tpr, label=f"AUC={auc:.3f}"); ax[0].plot([0, 1], [0, 1], "--")
ax[0].set(xlabel="False positive rate", ylabel="True positive rate", title="ROC"); ax[0].legend()
p, r, _ = precision_recall_curve(y_hold, s)
ax[1].plot(r, p, label=f"AP={ap:.3f}")
ax[1].set(xlabel="Recall", ylabel="Precision", title="Precision-Recall"); ax[1].legend()
plt.show()

cut = np.quantile(s, 0.90)
pred = (s >= cut).astype(int)
tn, fp, fn, tp = confusion_matrix(y_hold, pred).ravel()
prec, rec = tp / (tp + fp), tp / (tp + fn)
print(f"cut={cut:.4f} called={pred.sum()} precision={prec:.3f} recall={rec:.3f} "
      f"base={y_hold.mean():.3f} lift={prec / y_hold.mean():.2f}")
print("TP", tp, "FN", fn, "FP", fp, "TN", tn)

## Stage 8 · Сақтау

Таңдалған pipeline бүкіл `train.csv`-ге refit етілді. `model.joblib` ___ MB (<50 MB). Fresh процесте жүктеу: ___. `predictions.csv`: бағандар `id, y_pred, y_score`, ___ қатар, `y_pred=1` үлесі ___ %.

In [ ]:
import os
final_full = sklearn.base.clone(final)
final_full.fit(X, y)                       # бүкіл train.csv
joblib.dump(final_full, "model.joblib")
print("MB:", os.path.getsize("model.joblib") / 1e6)

X_test = test.drop(columns=["id"])[X.columns]
f = final_full.predict_proba if hasattr(final_full, "predict_proba") else None
sc_test = f(X_test)[:, 1] if f else final_full.decision_function(X_test)
cut_t = np.quantile(sc_test, 0.90)         # test-тің үздік 10%-ы
out = pd.DataFrame({"id": test["id"], "y_pred": (sc_test >= cut_t).astype(int),
                    "y_score": sc_test})
out.to_csv("predictions.csv", index=False)
print(out.shape, out["y_pred"].mean(), out["id"].nunique() == len(test))

# Model Card: Term Deposit Call Prioritisation
## 1. Мақсат
___ (кім үшін, 10% ереже)
## 2. Дерек
UCI Bank Marketing (Moro, Rita & Cortez, 2014; CC BY 4.0). train ___ қатар, "yes" ___ %; test ___ қатар. Бұл оқу деректері, нақты банк туралы қорытынды жасамаймын.
## 3. Pipeline
Preprocessing: ___. Модель: ___ (параметрлер ___). Валидация: StratifiedKFold(__), SEED=____.
## 4. Нәтиже
| Метрика | CV (dev) | Hold-out |
|---|---|---|
| ROC-AUC | ___ ± ___ | ___ |
| Average precision | ___ ± ___ | ___ |
| Precision@10% / Recall@10% / Lift | — | ___ / ___ / ___ |
## 5. Шектеулер және сәтсіздік жолдары
1. ___  2. ___  3. ___
## 6. Мониторинг
___ ("yes" үлесі, score таралуының дрейфі, шақырылған 10%-дағы нақты conversion)